# Basin Characteristics vs. Flood Return Periods

Evaluates relationships between physical basin attributes and LP3-derived flood return periods
at NWS action / flood / moderate / major thresholds.

**Sites included:** `record_ok=True`, `degenerate_fit=False`, `ppcc_ok=True`

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.nonparametric.smoothers_lowess as lowess_sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.inspection import permutation_importance
import shap

DATA_DIR = Path("/home/ryan/data/flood_hazard/ffa")
META_DIR = Path("/home/ryan/data/flood_hazard/metadata")
NHD_DIR  = Path("/home/ryan/data/flood_hazard/NHD/US SCS")

plt.rcParams['figure.dpi'] = 150
sns.set_theme(style='whitegrid', palette='muted')

RP_COLS = [
    'action_return_period_yr', 'flood_return_period_yr',
    'moderate_return_period_yr', 'major_return_period_yr'
]
THRESHOLDS = ['action', 'flood', 'moderate', 'major']

HUC2_NAMES = {
    '01': 'New England',        '02': 'Mid Atlantic',         '03': 'South Atlantic-Gulf',
    '04': 'Great Lakes',        '05': 'Ohio',                 '06': 'Tennessee',
    '07': 'Upper Mississippi',  '08': 'Lower Mississippi',    '09': 'Souris-Red-Rainy',
    '10': 'Missouri',           '11': 'Arkansas-White-Red',   '12': 'Texas-Gulf',
    '13': 'Rio Grande',         '14': 'Upper Colorado',       '15': 'Lower Colorado',
    '16': 'Great Basin',        '17': 'Pacific Northwest',    '18': 'California',
}

import matplotlib.font_manager as fm
font_path = '/home/ryan/data/assets/fonts/FiraSans/FiraSans-Regular.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'Fira Sans'

## 1. Load & Merge Data

In [ ]:
ffa  = pd.read_parquet(DATA_DIR / "flood_frequency.parquet").drop_duplicates('site_no')
ppcc = pd.read_parquet(DATA_DIR / "ppcc.parquet")
sq   = pd.read_parquet(DATA_DIR / "standard_quantiles.parquet")

ffa[RP_COLS] = ffa[RP_COLS].replace(np.inf, np.nan)

ok_sites = ppcc.loc[ppcc['ppcc_ok'], 'site_no']
ffa_ok = ffa[
    (ffa['record_ok'] == True) &
    (ffa['degenerate_fit'] == False) &
    ffa['site_no'].isin(ok_sites)
].copy()

si = pd.read_parquet(META_DIR / "site_info.parquet",
                     columns=['site_no','drainage_area_sqmi','elevation_ft',
                               'latitude','longitude','huc8','state_cd'])
cg = pd.read_parquet(META_DIR / "channel_geometry.parquet",
                     columns=['site_no','bankfull_width_ft','nhd_slope_ft_ft','bkfw_da_in_range'])
sp = pd.read_parquet(META_DIR / "stream_power.parquet")
gm = pd.read_parquet(META_DIR / "gage_map.parquet", columns=['site_no','reach_id'])

df = (ffa_ok
      .merge(sq,  on='site_no', how='left')
      .merge(si,  on='site_no', how='left')
      .merge(cg,  on='site_no', how='left')
      .merge(sp,  on='site_no', how='left')
      .merge(gm,  on='site_no', how='left'))

df['COMID'] = pd.to_numeric(df['reach_id'], errors='coerce').astype('Int64')

print(f"QC-pass sites : {len(df):,}")
print(f"With COMID    : {df['COMID'].notna().sum():,}")
print(f"With slope    : {df['nhd_slope_ft_ft'].notna().sum():,}")
print(f"With bankfull : {df['bankfull_width_ft'].notna().sum():,}")

## 2. Load NHD Basin Attributes and Join

In [ ]:
def load_nhd_table(files_spec, keep_cols):
    """Load and vertically concat NHD regional CSV files after renaming columns.
    files_spec: list of (path, rename_dict)
    """
    parts = []
    for path, renames in files_spec:
        d = pd.read_csv(path).rename(columns=renames)
        parts.append(d[[c for c in keep_cols if c in d.columns]])
    combined = pd.concat(parts, ignore_index=True)
    combined['COMID'] = pd.to_numeric(combined['COMID'], errors='coerce').astype('Int64')
    return combined.drop_duplicates('COMID')

SG = NHD_DIR / "Size_Gradient"
VC = NHD_DIR / "Valley_Confinement"
HY = NHD_DIR / "Hydrology"
TM = NHD_DIR / "Temperature"

# ── Size_Gradient ──────────────────────────────────────────────────────────────
sg_rename = {
    'flow_cfs': 'nhd_flow_cfs', 'Flow_cfs': 'nhd_flow_cfs',
    'StreamOrder': 'stream_order', 'StreamOrde': 'stream_order',
    'Size_Class': 'size_class', 'SizeClass': 'size_class',
    'Slope': 'nhd_slope_sg', 'slope': 'nhd_slope_sg',
    'Gradient_Class': 'gradient_class', 'GradientClass': 'gradient_class',
}
nhd_sg = load_nhd_table([
    (SG / "East_SizeGradient.csv", sg_rename),
    (SG / "West_SizeGradient.csv", sg_rename),
    (SG / "UM_SizeGradient.csv",   sg_rename),
    (SG / "LM_SizeGradient.csv",   sg_rename),
], keep_cols=['COMID', 'nhd_flow_cfs', 'stream_order', 'size_class', 'nhd_slope_sg', 'gradient_class'])
nhd_sg['nhd_slope_sg'] = pd.to_numeric(nhd_sg['nhd_slope_sg'], errors='coerce')
nhd_sg.loc[nhd_sg['nhd_slope_sg'] <= 0, 'nhd_slope_sg'] = np.nan

# ── Valley_Confinement ────────────────────────────────────────────────────────
vc_rename = {'RWA': 'SWA'}  # East uses RWA; others use SWA
nhd_vc = load_nhd_table([
    (VC / "East_VC.csv", vc_rename),
    (VC / "West_VC.csv", {}),
    (VC / "UM_VC.csv",   {}),
    (VC / "LM_VC.csv",   {}),
], keep_cols=['COMID', 'RL', 'VBA', 'SWA', 'VBA_RWA_R', 'Confinement'])
nhd_vc = nhd_vc[nhd_vc['Confinement'] != '#DIV/0!'].copy()
nhd_vc['VBA_RWA_R'] = pd.to_numeric(nhd_vc['VBA_RWA_R'], errors='coerce')

# ── Hydrology ────────────────────────────────────────────────────────────────
hy_rename = {'prop': 'propg2', 'prope1': 'prope'}
nhd_hy = load_nhd_table([
    (HY / "East_hydro_classes.csv", hy_rename),
    (HY / "West_hydro_classes.csv", hy_rename),
    (HY / "UM_hydro_classes.csv",   hy_rename),
    (HY / "LM_hydro_classes.csv",   hy_rename),
], keep_cols=['COMID', 'exp', 'prope', 'propg2', 'propg14', 'propg30'])

# ── Temperature ──────────────────────────────────────────────────────────────
tm_rename = {
    'Maheu_type': 'maheu_class', 'Maheu_class': 'maheu_class',
    'JulAug_tempC': 'julaug_tempC', 'JulyAug_tempC': 'julaug_tempC',
    'JulAug_Class': 'julaug_class', 'JulyAug_Class': 'julaug_class',
}
nhd_tm = load_nhd_table([
    (TM / "East_temp.csv", tm_rename),
    (TM / "West_temp.csv", tm_rename),
    (TM / "UM_temp.csv",   tm_rename),
    (TM / "LM_temp.csv",   tm_rename),
], keep_cols=['COMID', 'maheu_class', 'julaug_tempC', 'julaug_class'])
nhd_tm['julaug_tempC'] = pd.to_numeric(nhd_tm['julaug_tempC'], errors='coerce')

# ── Join all to df ────────────────────────────────────────────────────────────
for name, tbl in [('Size_Gradient', nhd_sg), ('Valley_Confinement', nhd_vc),
                  ('Hydrology', nhd_hy), ('Temperature', nhd_tm)]:
    n_before = df['COMID'].notna().sum()
    df = df.merge(tbl, on='COMID', how='left')
    n_joined = df[tbl.columns.drop('COMID')[0]].notna().sum()
    print(f"{name:22s}  rows joined: {n_joined:,} / {n_before:,}")

## 3. Feature Engineering

In [ ]:
# Log-transformed predictors
df['log_da']         = np.log10(df['drainage_area_sqmi'].clip(lower=1e-3))
df['log_slope']      = np.log10(df['nhd_slope_ft_ft'].clip(lower=1e-6))
df['log_sg_slope']   = np.log10(df['nhd_slope_sg'].where(df['nhd_slope_sg'] > 0).clip(lower=1e-6))
df['log_bkfw']       = np.log10(df['bankfull_width_ft'].clip(lower=1))
df['log_elev']       = np.log10(df['elevation_ft'].clip(lower=1))
df['log_ssp_action'] = np.log10(df['action_ssp_wm2'].clip(lower=0.01))
df['log_rl']         = np.log10(df['RL'].clip(lower=1e-3))
df['log_vba_swa_r']  = np.log10(df['VBA_RWA_R'].clip(lower=1e-3))

# Log-transformed targets
for t in THRESHOLDS:
    df[f'log_{t}_rp'] = np.log10(df[f'{t}_return_period_yr'])
df['log_q2']  = np.log10(df['q2_cfs'].clip(lower=1))
df['log_q10'] = np.log10(df['q10_cfs'].clip(lower=1))

# Derived variables
df['huc2']         = df['huc8'].astype('Int64').astype(str).str.zfill(8).str[:2]
df['huc2_name']    = df['huc2'].map(HUC2_NAMES)
df['specific_q2']  = df['q2_cfs'] / df['drainage_area_sqmi'].clip(lower=1e-3)
df['specific_q10'] = df['q10_cfs'] / df['drainage_area_sqmi'].clip(lower=1e-3)
df['q10_q2_ratio'] = df['q10_cfs'] / df['q2_cfs'].clip(lower=1)
df['log_q10_q2_r'] = np.log10(df['q10_q2_ratio'].clip(lower=1e-3))
df['log_spec_q2']  = np.log10(df['specific_q2'].clip(lower=1e-3))
df['da_slope_prod'] = df['log_da'] + df['log_slope']

print(f"Final dataframe: {df.shape}")
predictor_coverage = {
    'log_da': df['log_da'].notna().sum(),
    'log_slope': df['log_slope'].notna().sum(),
    'log_bkfw': df['log_bkfw'].notna().sum(),
    'log_elev': df['log_elev'].notna().sum(),
    'log_ssp_action': df['log_ssp_action'].notna().sum(),
    'log_rl': df['log_rl'].notna().sum(),
    'log_vba_swa_r': df['log_vba_swa_r'].notna().sum(),
    'julaug_tempC': df['julaug_tempC'].notna().sum(),
    'prope': df['prope'].notna().sum(),
    'propg2': df['propg2'].notna().sum(),
    'propg30': df['propg30'].notna().sum(),
    'Confinement': df['Confinement'].notna().sum(),
    'size_class': df['size_class'].notna().sum(),
}
cov = pd.Series(predictor_coverage, name='n_valid').to_frame()
cov['pct'] = (cov['n_valid'] / len(df) * 100).round(1)
print(cov.to_string())

## 4. Univariate Distributions

In [ ]:
# Return period targets
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
labels = ['Action', 'Flood', 'Moderate', 'Major']
for ax, t, lbl in zip(axes, THRESHOLDS, labels):
    col = f'{t}_return_period_yr'
    vals = df[col].dropna()
    ax.hist(np.log10(vals.clip(lower=0.1)), bins=40, color='steelblue', edgecolor='none', alpha=0.8)
    med = vals.median()
    ax.axvline(np.log10(med), color='firebrick', lw=1.5, label=f'Median={med:.1f} yr')
    ax.set_title(lbl)
    ax.set_xlabel('log₁₀(Return period, yr)')
    ax.legend(fontsize=7)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'10^{{{x:.0f}}}' if x != int(x) else f'{10**x:.0f}'))
axes[0].set_ylabel('Count')
fig.suptitle('Distribution of NWS Threshold Return Periods', fontweight='bold')
plt.tight_layout()
plt.show()

# Key predictor distributions
pred_info = [
    ('log_da',    'log₁₀(Drainage area, mi²)'),
    ('log_slope', 'log₁₀(Channel slope, ft/ft)'),
    ('log_bkfw',  'log₁₀(Bankfull width, ft)'),
    ('log_elev',  'log₁₀(Elevation, ft)'),
]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, (col, xlabel) in zip(axes, pred_info):
    vals = df[col].dropna()
    ax.hist(vals, bins=40, color='teal', edgecolor='none', alpha=0.8)
    ax.axvline(vals.median(), color='firebrick', lw=1.5)
    ax.set_xlabel(xlabel)
axes[0].set_ylabel('Count')
fig.suptitle('Distribution of Key Predictors', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Spearman Correlation Analysis

In [ ]:
PRED_COLS = ['log_da', 'log_slope', 'log_bkfw', 'log_elev', 'log_ssp_action',
             'log_rl', 'log_vba_swa_r', 'julaug_tempC', 'prope', 'propg2', 'propg30',
             'log_spec_q2', 'latitude', 'longitude']
TARGET_COLS = ['log_action_rp', 'log_flood_rp', 'log_moderate_rp', 'log_major_rp',
               'log_q10_q2_r', 'lp3_scale', 'lp3_weighted_skew']

all_cols = PRED_COLS + TARGET_COLS
mat = df[all_cols].dropna(how='all')

# Pairwise Spearman (ignores NaN via pairwise complete)
n_cols = len(all_cols)
rho_mat = np.full((n_cols, n_cols), np.nan)
for i, c1 in enumerate(all_cols):
    for j, c2 in enumerate(all_cols):
        if i == j:
            rho_mat[i, j] = 1.0
            continue
        pair = mat[[c1, c2]].dropna()
        if len(pair) > 10:
            rho, _ = stats.spearmanr(pair.iloc[:, 0].values, pair.iloc[:, 1].values)
            rho_mat[i, j] = float(rho)

rho_df = pd.DataFrame(rho_mat, index=all_cols, columns=all_cols)

# Heatmap: predictors vs targets only
sub = rho_df.loc[PRED_COLS, TARGET_COLS]
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(sub, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            linewidths=0.3, ax=ax, cbar_kws={'shrink': 0.7})
ax.set_title('Spearman Correlation: Basin Attributes vs. Flood Metrics', fontweight='bold')
plt.tight_layout()
plt.show()

# Top predictors per target
print("\nTop 8 predictors by |Spearman r| for each return period target:")
for tc in ['log_action_rp', 'log_flood_rp', 'log_moderate_rp', 'log_major_rp']:
    top = rho_df.loc[PRED_COLS, tc].abs().sort_values(ascending=False).head(8)
    print(f"\n{tc}:")
    for pred, r_abs in top.items():
        r_signed = rho_df.loc[pred, tc]
        print(f"  {pred:20s}  r = {r_signed:+.3f}")

## 6. Scatter Plots

In [ ]:
# 6a — Return period vs. drainage area (log-log), colored by thermal class
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)
labels = ['Action', 'Flood', 'Moderate', 'Major']
palette = sns.color_palette('tab10', n_colors=df['julaug_class'].nunique())
classes = sorted(df['julaug_class'].dropna().unique())
color_map = dict(zip(classes, palette))

for ax, t, lbl in zip(axes, THRESHOLDS, labels):
    sub = df[['log_da', f'{t}_return_period_yr', 'julaug_class']].dropna()
    for cls, grp in sub.groupby('julaug_class'):
        ax.scatter(grp['log_da'], np.log10(grp[f'{t}_return_period_yr']),
                   s=8, alpha=0.5, color=color_map.get(cls, 'gray'), label=cls)
    # LOWESS trend
    xy = sub[['log_da', f'{t}_return_period_yr']].dropna()
    if len(xy) > 20:
        xy_sorted = xy.sort_values('log_da')
        smoothed = lowess_sm.lowess(np.log10(xy_sorted[f'{t}_return_period_yr']),
                                     xy_sorted['log_da'], frac=0.3, return_sorted=True)
        ax.plot(smoothed[:, 0], smoothed[:, 1], color='black', lw=1.5, zorder=5)
    ax.set_title(lbl)
    ax.set_xlabel('log₁₀(DA, mi²)')
    ax.set_ylabel('log₁₀(Return period, yr)')
    
    ax.set_ylim(0, 5)

handles = [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=c, markersize=6, label=k)
           for k, c in color_map.items()]
axes[-1].legend(handles=handles, fontsize=6, title='Thermal class', title_fontsize=7)
fig.suptitle('Return Period vs. Drainage Area (colored by thermal class)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 6b — Return period vs. channel slope, colored by NHD size class
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)
size_classes = sorted(df['size_class'].dropna().unique())
sc_palette = sns.color_palette('tab20', n_colors=len(size_classes))
sc_map = dict(zip(size_classes, sc_palette))

for ax, t, lbl in zip(axes, THRESHOLDS, labels):
    sub = df[['log_slope', f'{t}_return_period_yr', 'size_class']].dropna()
    for cls, grp in sub.groupby('size_class'):
        ax.scatter(grp['log_slope'], np.log10(grp[f'{t}_return_period_yr']),
                   s=8, alpha=0.5, color=sc_map.get(cls, 'gray'), label=cls)
    xy = sub[['log_slope', f'{t}_return_period_yr']].dropna()
    if len(xy) > 20:
        xy_s = xy.sort_values('log_slope')
        sm = lowess_sm.lowess(np.log10(xy_s[f'{t}_return_period_yr']),
                               xy_s['log_slope'], frac=0.3, return_sorted=True)
        ax.plot(sm[:, 0], sm[:, 1], color='black', lw=1.5, zorder=5)
    ax.set_title(lbl)
    ax.set_xlabel('log₁₀(Channel slope, ft/ft)')
    ax.set_ylabel('log₁₀(Return period, yr)')
    ax.set_ylim(0,5)

handles = [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=c, markersize=6, label=k)
           for k, c in sc_map.items()]
axes[-1].legend(handles=handles, fontsize=5, title='Size class', title_fontsize=7, ncol=2)
fig.suptitle('Return Period vs. Channel Slope (colored by NHD size class)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 6c — Q10/Q2 ratio (flashiness proxy) vs. key predictors
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
scatter_pairs = [
    ('log_da',    'log₁₀(DA, mi²)'),
    ('log_slope', 'log₁₀(Slope, ft/ft)'),
    ('log_bkfw',  'log₁₀(Bankfull width, ft)'),
    ('julaug_tempC', 'Jul-Aug temp (°C)'),
    ('lp3_weighted_skew', 'LP3 weighted skew'),
]
for (xcol, xlabel), ax in zip(scatter_pairs, axes.flat[:5]):
    sub = df[[xcol, 'log_q10_q2_r']].dropna()
    ax.scatter(sub[xcol], sub['log_q10_q2_r'], s=6, alpha=0.4, color='steelblue')
    if len(sub) > 20:
        xy_s = sub.sort_values(xcol)
        sm = lowess_sm.lowess(xy_s['log_q10_q2_r'], xy_s[xcol], frac=0.3, return_sorted=True)
        ax.plot(sm[:, 0], sm[:, 1], color='firebrick', lw=1.5)
    rho, p = stats.spearmanr(sub[xcol], sub['log_q10_q2_r'])
    ax.set_xlabel(xlabel)
    ax.set_ylabel('log₁₀(Q10/Q2 ratio)')
    ax.set_title(f'r = {rho:.3f}', fontsize=9)

# Last panel: boxplot by Confinement
ax = axes.flat[5]
conf_order = ['Unconfined', 'Mod Confined', 'Confined']
conf_data = [df.loc[df['Confinement'] == c, 'log_q10_q2_r'].dropna() for c in conf_order]
ax.boxplot(conf_data, labels=[c.replace(' ', '\n') for c in conf_order],
           flierprops=dict(marker='.', markersize=3, alpha=0.3))
for i, (c, dat) in enumerate(zip(conf_order, conf_data), 1):
    ax.text(i, ax.get_ylim()[0], f'n={len(dat)}', ha='center', va='bottom', fontsize=7)
ax.set_ylabel('log₁₀(Q10/Q2 ratio)')
ax.set_title('By Confinement')

fig.suptitle('Q10/Q2 Ratio (Flashiness Proxy) vs. Basin Attributes', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 6d — LP3 scale and skew vs. DA×slope product, colored by gradient class
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
grad_classes = sorted(df['gradient_class'].dropna().unique())
gd_palette = sns.color_palette('viridis', n_colors=len(grad_classes))
gd_map = dict(zip(grad_classes, gd_palette))

for ax, ycol, ylabel in zip(axes,
    ['lp3_scale', 'lp3_weighted_skew'],
    ['LP3 scale (log₁₀ σ)', 'LP3 weighted skew (γ)']):
    sub = df[['da_slope_prod', ycol, 'gradient_class']].dropna()
    for cls, grp in sub.groupby('gradient_class'):
        ax.scatter(grp['da_slope_prod'], grp[ycol],
                   s=8, alpha=0.5, color=gd_map.get(cls, 'gray'), label=cls)
    xy_s = sub.sort_values('da_slope_prod')
    sm = lowess_sm.lowess(xy_s[ycol], xy_s['da_slope_prod'], frac=0.3, return_sorted=True)
    ax.plot(sm[:, 0], sm[:, 1], color='black', lw=1.5, zorder=5)
    rho, _ = stats.spearmanr(sub['da_slope_prod'], sub[ycol])
    ax.set_xlabel('log₁₀(DA) + log₁₀(Slope)  [proxy for discharge magnitude]')
    ax.set_ylabel(ylabel)
    ax.set_title(f'r = {rho:.3f}')
    ax.legend(fontsize=6, title='Gradient class', title_fontsize=7)

fig.suptitle('LP3 Parameters vs. DA × Slope Product', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 6e — Specific stream power vs. return period
ssp_cols = ['action_ssp_wm2', 'flood_ssp_wm2', 'moderate_ssp_wm2', 'major_ssp_wm2']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, t, ssp, lbl in zip(axes, THRESHOLDS, ssp_cols, labels):
    sub = df[[ssp, f'{t}_return_period_yr']].dropna()
    sub = sub[(sub[ssp] > 0) & (sub[f'{t}_return_period_yr'] > 0)]
    ax.scatter(np.log10(sub[ssp]), np.log10(sub[f'{t}_return_period_yr']),
               s=6, alpha=0.35, color='darkorange')
    if len(sub) > 20:
        xy_s = sub.sort_values(ssp)
        sm = lowess_sm.lowess(np.log10(xy_s[f'{t}_return_period_yr']),
                               np.log10(xy_s[ssp]), frac=0.3, return_sorted=True)
        ax.plot(sm[:, 0], sm[:, 1], color='black', lw=1.5)
    rho, _ = stats.spearmanr(np.log10(sub[ssp]),
                              np.log10(sub[f'{t}_return_period_yr']))
    ax.set_title(f'{lbl}  (r={rho:.2f})')
    ax.set_xlabel('log₁₀(SSP, W/m²)')
    ax.set_ylabel('log₁₀(Return period, yr)')
    ax.set_ylim(0,5)

fig.suptitle('Specific Stream Power vs. Return Period at Each NWS Threshold', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. HUC2 Regional Breakdown

In [ ]:
def huc2_boxplot(ax, col, title, df=df, log_scale=False):
    sub = df[['huc2_name', col]].dropna()
    order = sub.groupby('huc2_name')[col].median().sort_values().index
    data  = [sub.loc[sub['huc2_name'] == h, col].values for h in order]
    ax.boxplot(data, vert=True, patch_artist=True, notch=False,
               flierprops=dict(marker='.', markersize=2, alpha=0.3),
               medianprops=dict(color='firebrick', lw=1.5))
    ax.set_xticks(range(1, len(order) + 1))
    ax.set_xticklabels([f"{h}\n(n={len(d)})" for h, d in zip(order, data)],
                       rotation=45, ha='right', fontsize=6)
    ax.set_title(title)
    if log_scale:
        ax.set_yscale('log')
    
    ax.set_ylim(0,50)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
huc2_boxplot(axes[0, 0], 'action_return_period_yr', 'Action RP by HUC2', log_scale=True)
axes[0, 0].set_ylabel('Return period (yr)')
huc2_boxplot(axes[0, 1], 'flood_return_period_yr', 'Flood RP by HUC2', log_scale=True)
axes[0, 1].set_ylabel('Return period (yr)')
huc2_boxplot(axes[1, 0], 'lp3_scale', 'LP3 Scale by HUC2')
axes[1, 0].set_ylabel('LP3 scale (log₁₀ σ)')
huc2_boxplot(axes[1, 1], 'lp3_weighted_skew', 'LP3 Weighted Skew by HUC2')
axes[1, 1].set_ylabel('Weighted skew (γ)')

fig.suptitle('Return Period and LP3 Parameters by HUC2 Region', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Return period distributions by gradient class and confinement class
grad_order = ['Very low', 'Low', 'Moderate', 'Moderate High', 'High', 'Steep']
grad_order = [g for g in grad_order if g in df['gradient_class'].values]
grad_palette = dict(zip(grad_order, sns.color_palette('viridis', n_colors=len(grad_order))))

conf_order = ['Unconfined', 'Mod Confined', 'Confined']
conf_order = [c for c in conf_order if c in df['Confinement'].values]
conf_palette = dict(zip(conf_order, sns.color_palette('Set2', n_colors=len(conf_order))))

fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey='row')
labels = ['Action', 'Flood', 'Moderate', 'Major']

for col, (t, lbl) in enumerate(zip(THRESHOLDS, labels)):
    rp_col = f'{t}_return_period_yr'

    # Row 0: gradient class
    ax = axes[0, col]
    for cls in grad_order:
        vals = np.log10(df.loc[df['gradient_class'] == cls, rp_col].dropna().clip(lower=0.1))
        if len(vals) > 5:
            sns.kdeplot(vals, ax=ax, color=grad_palette[cls], label=cls, linewidth=1.5)
    ax.set_xlabel('log₁₀(Return period, yr)')
    ax.set_title(lbl)
    if col == 0:
        ax.set_ylabel('Density')
        ax.legend(fontsize=6, title='Gradient class', title_fontsize=7)

    # Row 1: confinement class
    ax = axes[1, col]
    for cls in conf_order:
        vals = np.log10(df.loc[df['Confinement'] == cls, rp_col].dropna().clip(lower=0.1))
        if len(vals) > 5:
            sns.kdeplot(vals, ax=ax, color=conf_palette[cls], label=cls, linewidth=1.5,
                        fill=True, alpha=0.2)
    ax.set_xlabel('log₁₀(Return period, yr)')
    if col == 0:
        ax.set_ylabel('Density')
        ax.legend(fontsize=7, title='Confinement', title_fontsize=7)

axes[0, 0].annotate('Gradient class', xy=(0, 0.5), xycoords='axes fraction',
                    rotation=90, va='center', ha='right', fontsize=8, fontweight='bold',
                    xytext=(-38, 0), textcoords='offset points')
axes[1, 0].annotate('Confinement', xy=(0, 0.5), xycoords='axes fraction',
                    rotation=90, va='center', ha='right', fontsize=8, fontweight='bold',
                    xytext=(-38, 0), textcoords='offset points')

fig.suptitle('Return Period Distributions by Gradient Class and Confinement',
             fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey='row')

for col, (t, lbl) in enumerate(zip(THRESHOLDS, labels)):
    rp_col = f'{t}_return_period_yr'

    # Row 0: gradient class
    ax = axes[0, col]
    data = [df.loc[df['gradient_class'] == cls, rp_col].dropna().values for cls in grad_order]
    bp = ax.boxplot(data, patch_artist=True,
                    flierprops=dict(marker='.', markersize=2, alpha=0.3),
                    medianprops=dict(color='black', lw=1.5))
    for patch, cls in zip(bp['boxes'], grad_order):
        patch.set_facecolor(grad_palette[cls])
        patch.set_alpha(0.7)
    ax.set_yscale('log')
    ax.set_xticks(range(1, len(grad_order) + 1))
    ax.set_xticklabels([f"{cls}\n(n={len(d)})" for cls, d in zip(grad_order, data)],
                       fontsize=6, rotation=30, ha='right')
    ax.set_title(lbl)
    if col == 0:
        ax.set_ylabel('Return period (yr)')
    if col == 3:
        handles = [plt.Rectangle((0, 0), 1, 1, facecolor=grad_palette[cls], alpha=0.7)
                   for cls in grad_order]
        ax.legend(handles, grad_order, fontsize=6, title='Gradient class',
                  title_fontsize=7, loc='lower left')
    
    ax.set_ylim(0, 1000)

    # Row 1: confinement class
    ax = axes[1, col]
    data = [df.loc[df['Confinement'] == cls, rp_col].dropna().values for cls in conf_order]
    bp = ax.boxplot(data, patch_artist=True,
                    flierprops=dict(marker='.', markersize=2, alpha=0.3),
                    medianprops=dict(color='black', lw=1.5))
    for patch, cls in zip(bp['boxes'], conf_order):
        patch.set_facecolor(conf_palette[cls])
        patch.set_alpha(0.7)
    ax.set_yscale('log')
    ax.set_xticks(range(1, len(conf_order) + 1))
    ax.set_xticklabels([f"{cls}\n(n={len(d)})" for cls, d in zip(conf_order, data)],
                       fontsize=7, rotation=30, ha='right')
    if col == 0:
        ax.set_ylabel('Return period (yr)')
    if col == 3:
        handles = [plt.Rectangle((0, 0), 1, 1, facecolor=conf_palette[cls], alpha=0.7)
                   for cls in conf_order]
        ax.legend(handles, conf_order, fontsize=7, title='Confinement',
                  title_fontsize=7, loc='lower left')
    
    ax.set_ylim(0, 500)

axes[0, 0].annotate('Gradient class', xy=(0, 0.5), xycoords='axes fraction',
                    rotation=90, va='center', ha='right', fontsize=8, fontweight='bold',
                    xytext=(-48, 0), textcoords='offset points')
axes[1, 0].annotate('Confinement', xy=(0, 0.5), xycoords='axes fraction',
                    rotation=90, va='center', ha='right', fontsize=8, fontweight='bold',
                    xytext=(-48, 0), textcoords='offset points')

fig.suptitle('Return Period Boxplots by Gradient Class and Confinement',
             fontweight='bold')
plt.tight_layout()
plt.show()

## 8. OLS Regression

In [ ]:
# 8a — Progressive OLS for log_action_rp
models_spec = [
    ('M1', 'log_action_rp ~ log_da',
     ['log_action_rp', 'log_da']),
    ('M2', 'log_action_rp ~ log_da + log_slope',
     ['log_action_rp', 'log_da', 'log_slope']),
    ('M3', 'log_action_rp ~ log_da + log_slope + log_bkfw + log_elev + julaug_tempC + prope',
     ['log_action_rp', 'log_da', 'log_slope', 'log_bkfw', 'log_elev', 'julaug_tempC', 'prope']),
    ('M4', 'log_action_rp ~ log_da + log_slope + log_bkfw + log_elev + julaug_tempC + prope + C(Confinement)',
     ['log_action_rp', 'log_da', 'log_slope', 'log_bkfw', 'log_elev', 'julaug_tempC', 'prope', 'Confinement']),
]

results = {}
comparison_rows = []
for name, formula, used_cols in models_spec:
    fit_df = df[used_cols].dropna()
    fit_df = fit_df[fit_df['Confinement'] != '#DIV/0!'] if 'Confinement' in used_cols else fit_df
    res = smf.ols(formula, data=fit_df).fit()
    results[name] = res
    comparison_rows.append({
        'Model': name, 'n': len(fit_df),
        'R²': round(res.rsquared, 4),
        'adj-R²': round(res.rsquared_adj, 4),
        'AIC': round(res.aic, 1),
        'BIC': round(res.bic, 1),
    })

comparison = pd.DataFrame(comparison_rows).set_index('Model')
print("Model comparison — target: log_action_rp")
print(comparison.to_string())

print("\nBest model (M4) coefficients:")
print(results['M4'].summary2().tables[1].round(4))

In [ ]:
# 8b — Repeat progressive OLS for all four return period targets (M3 only)
m3_template = '{target} ~ log_da + log_slope + log_bkfw + log_elev + julaug_tempC + prope'
m3_cols = ['log_da', 'log_slope', 'log_bkfw', 'log_elev', 'julaug_tempC', 'prope']

rp_results = []
for t in THRESHOLDS:
    target = f'log_{t}_rp'
    sub = df[[target] + m3_cols].dropna()
    res = smf.ols(m3_template.format(target=target), data=sub).fit()
    rp_results.append({'Threshold': t, 'n': len(sub),
                        'R²': round(res.rsquared, 4),
                        'adj-R²': round(res.rsquared_adj, 4)})

print("M3 OLS results by threshold:")
print(pd.DataFrame(rp_results).set_index('Threshold').to_string())

# 8c — OLS for LP3 parameters
print("\nM3 OLS for LP3 parameters:")
for param in ['lp3_scale', 'lp3_weighted_skew']:
    sub = df[[param] + m3_cols].dropna()
    res = smf.ols(f'{param} ~ log_da + log_slope + log_bkfw + log_elev + julaug_tempC + prope',
                  data=sub).fit()
    print(f"\n{param}: n={len(sub)}, R²={res.rsquared:.4f}, adj-R²={res.rsquared_adj:.4f}")
    print(res.params.round(4).to_string())

In [ ]:
# 8d — M3 residual diagnostics by HUC2
sub = df[['log_action_rp', 'huc2_name'] + m3_cols].dropna()
res_m3 = smf.ols(
    'log_action_rp ~ log_da + log_slope + log_bkfw + log_elev + julaug_tempC + prope',
    data=sub
).fit()
sub = sub.copy()
sub['resid'] = res_m3.resid

order = sub.groupby('huc2_name')['resid'].median().sort_values().index
data  = [sub.loc[sub['huc2_name'] == h, 'resid'].values for h in order]

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.boxplot(data, patch_artist=True,
           flierprops=dict(marker='.', markersize=2, alpha=0.3),
           medianprops=dict(color='firebrick', lw=1.5))
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xticks(range(1, len(order) + 1))
ax.set_xticklabels([f"{h}\n(n={len(d)})" for h, d in zip(order, data)],
                    rotation=45, ha='right', fontsize=6)
ax.set_ylabel('M3 Residual (log₁₀ RP units)')
ax.set_title('M3 Residuals by HUC2 — unmodeled regional structure', fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Global M3 R²={res_m3.rsquared:.4f}, RMSE={res_m3.resid.std():.4f} log₁₀ units")

## 9. Random Forest Feature Importance

In [ ]:
# RF_FEATURES = ['log_da', 'log_slope', 'log_bkfw', 'log_elev', 'log_ssp_action',
#                'log_rl', 'log_vba_swa_r', 'julaug_tempC', 'prope', 'propg2', 'propg30',
#                'log_spec_q2', 'latitude', 'longitude']
# rf_target = 'log_action_rp'

# rf_df = df[[rf_target] + RF_FEATURES].dropna()
# X = rf_df[RF_FEATURES].values
# y = rf_df[rf_target].values
# print(f"RF dataset: {X.shape[0]:,} sites × {X.shape[1]} features")

# rf = RandomForestRegressor(
#     n_estimators=500, max_features='sqrt',
#     min_samples_leaf=10, random_state=42, n_jobs=-1
# )
# cv_r2 = cross_val_score(rf, X, y, cv=5, scoring='r2')
# print(f"5-fold CV R² = {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")

# rf.fit(X, y)
# print(f"In-sample R² = {rf.score(X, y):.3f}")

In [ ]:
# # MDI vs. permutation importance side-by-side
# mdi = rf.feature_importances_
# perm = permutation_importance(rf, X, y, n_repeats=20, random_state=42, n_jobs=-1)

# order_mdi  = np.argsort(mdi)
# order_perm = np.argsort(perm.importances_mean)

# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
# feat_labels = RF_FEATURES

# ax1.barh([feat_labels[i] for i in order_mdi], mdi[order_mdi], color='steelblue')
# ax1.set_title('MDI Feature Importance')
# ax1.set_xlabel('Importance')

# ax2.barh([feat_labels[i] for i in order_perm], perm.importances_mean[order_perm],
#          xerr=perm.importances_std[order_perm], color='darkorange', capsize=3)
# ax2.set_title('Permutation Importance (20 repeats)')
# ax2.set_xlabel('Mean decrease in R²')
# ax2.axvline(0, color='gray', lw=0.8)

# fig.suptitle('Random Forest — Feature Importances (target: log_action_rp)', fontweight='bold')
# plt.tight_layout()
# plt.show()

In [ ]:
# # SHAP summary
# explainer   = shap.TreeExplainer(rf)
# shap_values = explainer.shap_values(X)

# shap.summary_plot(shap_values, X, feature_names=RF_FEATURES,
#                   plot_type='dot', max_display=14, show=True)

## 10. Threshold Spacing Analysis

In [ ]:
# 10a — Consecutive threshold RP ratios vs. basin predictors
df['ratio_flood_action']    = df['flood_return_period_yr']    / df['action_return_period_yr'].clip(lower=0.01)
df['ratio_mod_flood']       = df['moderate_return_period_yr'] / df['flood_return_period_yr'].clip(lower=0.01)
df['ratio_major_mod']       = df['major_return_period_yr']    / df['moderate_return_period_yr'].clip(lower=0.01)

ratio_cols = ['ratio_flood_action', 'ratio_mod_flood', 'ratio_major_mod']
ratio_labels = ['Flood / Action', 'Moderate / Flood', 'Major / Moderate']

pred_for_ratio = ['log_da', 'log_slope', 'lp3_scale', 'lp3_weighted_skew']
fig, axes = plt.subplots(len(pred_for_ratio), len(ratio_cols), figsize=(13, 12), sharey='row')

for row, pred in enumerate(pred_for_ratio):
    for col, (rc, rl) in enumerate(zip(ratio_cols, ratio_labels)):
        ax = axes[row, col]
        sub = df[[pred, rc]].dropna()
        sub = sub[sub[rc] > 0]
        ax.scatter(sub[pred], np.log10(sub[rc]), s=5, alpha=0.3, color='steelblue')
        if len(sub) > 20:
            xy_s = sub.sort_values(pred)
            sm = lowess_sm.lowess(np.log10(xy_s[rc]), xy_s[pred], frac=0.3, return_sorted=True)
            ax.plot(sm[:, 0], sm[:, 1], color='firebrick', lw=1.5)
        rho, _ = stats.spearmanr(sub[pred], np.log10(sub[rc]))
        if row == 0:
            ax.set_title(rl, fontweight='bold')
        if col == 0:
            ax.set_ylabel(f'log₁₀({rc.split("_", 1)[1].replace("_", "/")})')
        ax.set_xlabel(pred)
        ax.text(0.97, 0.03, f'r={rho:.2f}', transform=ax.transAxes,
                ha='right', va='bottom', fontsize=7)

fig.suptitle('Consecutive Threshold Return Period Ratios vs. Basin Attributes',
             fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 10b — Facet grid: RP vs. log_da by threshold
melt_cols = ['site_no', 'log_da'] + RP_COLS
long = df[melt_cols].melt(id_vars=['site_no', 'log_da'],
                           value_vars=RP_COLS,
                           var_name='threshold', value_name='return_period_yr')
long['threshold'] = long['threshold'].str.replace('_return_period_yr', '', regex=False)
long['log_rp'] = np.log10(long['return_period_yr'])
long = long.dropna(subset=['log_da', 'log_rp'])

g = sns.FacetGrid(long, col='threshold', col_order=THRESHOLDS, height=3.5, aspect=1.1,
                  sharey=True)
g.map(sns.scatterplot, 'log_da', 'log_rp', alpha=0.2, s=6, color='steelblue')
g.set_axis_labels('log₁₀(DA, mi²)', 'log₁₀(Return period, yr)')
g.set_titles(col_template='{col_name}')
g.fig.suptitle('Return Period vs. Drainage Area by Threshold Level', y=1.02, fontweight='bold')
plt.show()

# 10c — Standard quantile vs. NWS threshold RP
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for ax, qcol, tcol, qlbl, tlbl in [
    (axes[0], 'q2_cfs',  'action_return_period_yr',   'Q2 (cfs)',  'Action RP (yr)'),
    (axes[1], 'q10_cfs', 'moderate_return_period_yr', 'Q10 (cfs)', 'Moderate RP (yr)'),
]:
    sub = df[[qcol, tcol, 'size_class']].dropna()
    sub = sub[(sub[qcol] > 0) & (sub[tcol] > 0)]
    for cls, grp in sub.groupby('size_class'):
        ax.scatter(np.log10(grp[qcol]), np.log10(grp[tcol]),
                   s=8, alpha=0.5, color=sc_map.get(cls, 'gray'), label=cls)
    ax.set_xlabel(f'log₁₀({qlbl})')
    ax.set_ylabel(f'log₁₀({tlbl})')
    ax.legend(fontsize=6, title='Size class', title_fontsize=7)

axes[0].set_title('LP3 Q2 vs. Action RP')
axes[1].set_title('LP3 Q10 vs. Moderate RP')
fig.suptitle('Standard LP3 Quantiles vs. NWS Threshold Return Periods', fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Summary Correlation Table

In [ ]:
target_rp_cols = [f'log_{t}_rp' for t in THRESHOLDS]

summary_rows = []
for pred in PRED_COLS:
    row = {'predictor': pred}
    abs_rs = []
    for tc in target_rp_cols:
        pair = df[[pred, tc]].dropna()
        if len(pair) > 10:
            r, _ = stats.spearmanr(pair[pred], pair[tc])
        else:
            r = np.nan
        row[tc.replace('log_', '').replace('_rp', '')] = round(r, 3)
        abs_rs.append(abs(r) if not np.isnan(r) else 0)
    row['mean_|r|'] = round(np.mean(abs_rs), 3)
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index('predictor').sort_values('mean_|r|', ascending=False)
print("Spearman correlations with NWS threshold return periods:")
print(summary.to_string())